# 机器翻译质量评估（BLEU）——英文与中文

BLEU（Bilingual Evaluation Understudy）是机器翻译最常用的自动评估指标：统计候选译文与参考译文之间 **n-gram 的重合度**，并加入长度惩罚（BP），得分区间 [0, 1]，越高表示翻译质量越好。

本 notebook 分别评估**英文**与**中文**的翻译质量：
- 英文：按空格分词，使用 `nltk.sentence_bleu`
- 中文：先用 `jieba` 分词，再使用 `nltk.sentence_bleu`


## 0. 环境准备

需要 `nltk` 与 `jieba`。若未安装，取消下面一行的注释运行（或直接在终端执行）：


In [1]:
# !pip install nltk jieba -i https://pypi.tuna.tsinghua.edu.cn/simple

from nltk.translate.bleu_score import sentence_bleu
import jieba
print("依赖就绪")


依赖就绪


## 1. 评估英文翻译质量

定义 `bleu_en`：按空格分词并转小写（英文适用）。`max_n` 决定使用的 n-gram 阶数上限，权重为各阶 n-gram 的平均。


In [2]:
def bleu_en(reference_str, candidate_str, max_n=4):
    reference = [reference_str.lower().split()]  # list of list of tokens，可放多个参考译文
    candidate = candidate_str.lower().split()    # list of tokens
    print("Reference:", reference)
    print("Candidate:", candidate)
    # 各阶 n-gram 权重平均分配：max_n=2 -> (0.5, 0.5)，max_n=4 -> (0.25, 0.25, 0.25, 0.25)
    weights = tuple([1.0 / max_n] * max_n)
    return sentence_bleu(reference, candidate, weights=weights)


### 1.1 英文示例（max_n=2）

参考译文：`the cat is on the mat`；候选译文：`the cat the cat the cat on the mat`（存在重复与漏译）。


In [3]:
ref = "the cat is on the mat"
cand = "the cat the cat the cat on the mat"
score_en = bleu_en(ref, cand, max_n=2)
print("BLEU:", score_en)   # 实测输出 0.4564


Reference: [['the', 'cat', 'is', 'on', 'the', 'mat']]
Candidate: ['the', 'cat', 'the', 'cat', 'the', 'cat', 'on', 'the', 'mat']
BLEU: 0.45643546458763845


### 1.2 权重构造与不同 n-gram 阶数的对比

`max_n` 越大，对长片段匹配的要求越严格，分数通常越低（候选译文没有与参考一致的 3-gram / 4-gram 时分数会趋于 0）。


In [4]:
print("max_n=2 权重:", tuple([1.0 / 2] * 2))
print("max_n=4 权重:", tuple([1.0 / 4] * 4))

for n in [1, 2, 4]:
    s = bleu_en(ref, cand, max_n=n)
    print(f"max_n={n} -> BLEU: {s:.4f}\n")


max_n=2 权重: (0.5, 0.5)
max_n=4 权重: (0.25, 0.25, 0.25, 0.25)
Reference: [['the', 'cat', 'is', 'on', 'the', 'mat']]
Candidate: ['the', 'cat', 'the', 'cat', 'the', 'cat', 'on', 'the', 'mat']
max_n=1 -> BLEU: 0.5556

Reference: [['the', 'cat', 'is', 'on', 'the', 'mat']]
Candidate: ['the', 'cat', 'the', 'cat', 'the', 'cat', 'on', 'the', 'mat']
max_n=2 -> BLEU: 0.4564

Reference: [['the', 'cat', 'is', 'on', 'the', 'mat']]
Candidate: ['the', 'cat', 'the', 'cat', 'the', 'cat', 'on', 'the', 'mat']
max_n=4 -> BLEU: 0.0000



C:\Users\叶湘伦\AppData\Local\Programs\Python\Python311\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


## 2. 评估中文翻译质量

中文没有天然的空格分词，需要先用 `jieba` 切词，再计算 BLEU。


In [5]:
def bleu_ch(ref_zh, cand_zh, max_n=4):
    reference_zh = [jieba.lcut(ref_zh)]   # 参考译文用 jieba 分词
    candidate_zh = jieba.lcut(cand_zh)    # 候选译文用 jieba 分词
    print("Reference:", reference_zh)
    print("Candidate:", candidate_zh)
    weights = tuple([1.0 / max_n] * max_n)
    return sentence_bleu(reference_zh, candidate_zh, weights=weights)


### 2.1 中文示例

参考译文：`猫在垫子上`；候选译文：`猫坐在垫子垫子垫子上`（多了主语"坐"且"垫子"重复）。


In [6]:
ref_zh = "猫在垫子上"
cand_zh = "猫坐在垫子垫子垫子上"

score_ch = bleu_ch(ref_zh, cand_zh, max_n=1)
print("BLEU(max_n=1):", score_ch)   # 实测输出 0.5

score_ch2 = bleu_ch(ref_zh, cand_zh, max_n=2)
print("BLEU(max_n=2):", score_ch2)  # 实测输出 0.3162


Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\叶湘伦\AppData\Local\Temp\jieba.cache
Loading model cost 0.465 seconds.
Prefix dict has been built successfully.


Reference: [['猫', '在', '垫子', '上']]
Candidate: ['猫', '坐在', '垫子', '垫子', '垫子', '上']
BLEU(max_n=1): 0.5
Reference: [['猫', '在', '垫子', '上']]
Candidate: ['猫', '坐在', '垫子', '垫子', '垫子', '上']
BLEU(max_n=2): 0.316227766016838


## 3. 平滑处理与多参考译文

候选译文与参考译文在高阶 n-gram 上完全无重叠时，BLEU 会直接为 0（如上例 max_n=4）。实际评估常用**平滑策略**（如 NLTK 的 `SmoothingFunction`）缓解这一问题；同时，提供**多个参考译文**也能提升评估的合理性。


In [7]:
from nltk.translate.bleu_score import SmoothingFunction
import numpy as np

smooth = SmoothingFunction().method1

multi_ref = [jieba.lcut("猫在垫子上"), jieba.lcut("猫在垫子上面")]
cand_zh2 = jieba.lcut("猫坐在垫子垫子上")

score_smooth = sentence_bleu(multi_ref, cand_zh2, weights=(0.25, 0.25, 0.25, 0.25),
                             smoothing_function=smooth)
print("多参考+平滑 BLEU(4-gram):", round(score_smooth, 4))

score_raw = sentence_bleu(multi_ref, cand_zh2, weights=(0.25, 0.25, 0.25, 0.25))
print("不加平滑 BLEU(4-gram):", round(score_raw, 4))


多参考+平滑 BLEU(4-gram): 0.1257
不加平滑 BLEU(4-gram): 0.0


C:\Users\叶湘伦\AppData\Local\Programs\Python\Python311\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


## 4. 结果解读

| 示例 | max_n | BLEU（实测） | 说明 |
| --- | --- | --- | --- |
| 英文 the cat... | 2 | 0.4564 | 候选有重复词与漏译，分数中等 |
| 英文 the cat... | 4 | ≈0 | 无 4-gram 重叠，趋于 0 |
| 中文 猫在垫子上 | 1 | 0.5000 | 词匹配一半（垫子重复拉低分母） |
| 中文 猫在垫子上 | 2 | 0.3162 | 二元词组匹配进一步下降 |

**要点**：BLEU 分数的高低受 n-gram 阶数、平滑策略、参考译文数量影响；对比不同系统时应保持参数一致。BLEU 与人工评估存在相关性，但无法完全替代人工判断（不衡量语义与流畅度）。
